# Question 10 - VAE Reparameterization Step (Fashion-MNIST)
### 1. Formula
$$z = \mu + \sigma\odot\epsilon,\qquad \sigma = e^{\frac{1}{2}\log\sigma^2},\qquad \epsilon\sim\mathcal{N}(0,I)$$

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
def reparameterize(mu, logvar):
    eps = tf.random.normal(tf.shape(mu))
    return mu + tf.exp(0.5 * logvar) * eps

In [ ]:
# Use it inside a small encoder on real Fashion-MNIST images
(x_train, _), _ = keras.datasets.fashion_mnist.load_data()
x = x_train[:32].astype("float32") / 255.0

class Encoder(keras.Model):
    def __init__(self, latent_dim=8):
        super().__init__()
        self.net = keras.Sequential([layers.Flatten(), layers.Dense(256, activation='relu')])
        self.mu = layers.Dense(latent_dim)
        self.logvar = layers.Dense(latent_dim)
    def call(self, x):
        h = self.net(x)
        return self.mu(h), self.logvar(h)

enc = Encoder()
with tf.GradientTape() as tape:
    mu, logvar = enc(x)
    z = reparameterize(mu, logvar)
    loss = tf.reduce_mean(z ** 2)
grads = tape.gradient(loss, enc.trainable_variables)

print("mu:", mu.shape, " logvar:", logvar.shape, " z:", z.shape)
print("All encoder gradients exist:", all(g is not None for g in grads))